# 01 Import and Debitor Matrix

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
from IPython.display import display

ARTIFACT_DIR = Path.cwd() / "notebook_artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
file_path = "../data/raw/UDB - 2010 - 2025 - prep - complete.xlsx"
df = pd.read_excel(file_path)

print("Zeilen:", df.shape[0])
print("Spalten:", df.shape[1])

Zeilen: 15262
Spalten: 14


In [3]:
df.columns.tolist()

['ID',
 'Datum',
 'Rechnungsnummer',
 'Vertragsart',
 'Umsatzart',
 'Umsatzart_Sub',
 'Jobkategorie',
 'Branche',
 'Debitor',
 'Netto_Betrag_Indexed',
 'Nebenkosten_Indexed',
 'Placement',
 'OTE',
 'Fuehrungsposition']

In [4]:
df.head()

,ID,Datum,Rechnungsnummer,Vertragsart,Umsatzart,Umsatzart_Sub,Jobkategorie,Branche,Debitor,Netto_Betrag_Indexed,Nebenkosten_Indexed,Placement,OTE,Fuehrungsposition
0,854,2010-08-09,E-PCVII-10-7036,Prozent - 3 Raten,1 Personalberatung,Start,Spezialist (Kerngeschäft),Industrial,Debitor 377,0.051282,0.0,False,NaN,False
1,860,2010-08-12,E-PCII-10-2001,Prozent - 3 Raten,1 Personalberatung,Start,C-Level,Industrial,Debitor 547,0.055556,0.0,False,NaN,False
2,861,2010-08-12,E-PCI-10-1085,Pauschalhonorar,1 Personalberatung,Erfolgsauftrag,C-Level,Digital & Technology,Debitor 419,0.235043,0.0,True,NaN,False
3,862,2010-08-13,E-PCI-10-1086,Prozent - 3 Raten,1 Personalberatung,Start,Vertrieb/Presales/Marketing,Digital & Technology,Debitor 185,0.056974,0.0,False,NaN,False
4,863,2010-08-13,E-PCI-10-1087,Prozent - 3 Raten,1 Personalberatung,Start,Vertrieb/Presales/Marketing,Digital & Technology,Debitor 185,0.056974,0.0,False,NaN,False


In [5]:
# Datumsfeld in ein konsistentes Format überführen

df["Datum"] = pd.to_datetime(df["Datum"], errors="coerce")

print("Frühestes Datum:", df["Datum"].min())
print("Spätestes Datum:", df["Datum"].max())
print("Fehlende Datumswerte:", df["Datum"].isna().sum())

Frühestes Datum: 2010-01-04 00:00:00
Spätestes Datum: 2025-12-31 00:00:00
Fehlende Datumswerte: 0


In [6]:
print("Placement:")
print(df["Placement"].value_counts(dropna=False))
print()

print("Fuehrungsposition:")
print(df["Fuehrungsposition"].value_counts(dropna=False))

Placement:
Placement
False    11373
True      3889
Name: count, dtype: int64

Fuehrungsposition:
Fuehrungsposition
False    12230
True      3032
Name: count, dtype: int64


In [7]:
def get_mode(series):
    series = series.dropna().astype(str).str.strip()
    if series.empty:
        return np.nan
    return series.mode().iloc[0]


def count_distinct(series):
    series = series.dropna().astype(str).str.strip()
    series = series[series != ""]
    return series.nunique()

In [8]:
# Debitor-Feature-Matrix aufbauen

debitor_features = df.groupby("Debitor").agg(
    invoice_count=("Rechnungsnummer", "nunique"),
    first_invoice_date=("Datum", "min"),
    last_invoice_date=("Datum", "max"),
    active_years_count=("Datum", lambda x: x.dt.year.nunique()),
    net_index_sum=("Netto_Betrag_Indexed", "sum"),
    net_index_mean=("Netto_Betrag_Indexed", "mean"),
    nk_index_sum=("Nebenkosten_Indexed", "sum"),
    nk_index_mean=("Nebenkosten_Indexed", "mean"),
    placement_count=("Placement", lambda x: x.fillna(False).sum()),
    placement_rate=("Placement", lambda x: x.fillna(False).mean()),
    leadership_count=("Fuehrungsposition", lambda x: x.fillna(False).sum()),
    leadership_rate=("Fuehrungsposition", lambda x: x.fillna(False).mean()),
    dominant_contract_type=("Vertragsart", get_mode),
    dominant_revenue_subtype=("Umsatzart_Sub", get_mode),
    dominant_job_category=("Jobkategorie", get_mode),
    dominant_industry=("Branche", get_mode),
    contract_type_nunique=("Vertragsart", count_distinct),
    revenue_subtype_nunique=("Umsatzart_Sub", count_distinct),
    job_category_nunique=("Jobkategorie", count_distinct),
    industry_nunique=("Branche", count_distinct)
).reset_index()

In [9]:
# Zusätzliche Zeitmerkmale ergänzen

debitor_features["active_days"] = (
    debitor_features["last_invoice_date"] - debitor_features["first_invoice_date"]
).dt.days

analysis_date = pd.Timestamp("2025-12-31")

debitor_features["recency_days"] = (
    analysis_date - debitor_features["last_invoice_date"]
).dt.days

In [10]:
print("Anzahl Debitoren:", debitor_features.shape[0])
print("Anzahl Features:", debitor_features.shape[1])

Anzahl Debitoren: 1364
Anzahl Features: 23


In [11]:
debitor_features.head()

,Debitor,invoice_count,first_invoice_date,last_invoice_date,active_years_count,net_index_sum,net_index_mean,nk_index_sum,nk_index_mean,placement_count,...,dominant_contract_type,dominant_revenue_subtype,dominant_job_category,dominant_industry,contract_type_nunique,revenue_subtype_nunique,job_category_nunique,industry_nunique,active_days,recency_days
0,Debitor 1,101,2010-11-25,2025-11-27,16,5.974359,0.059152,0.581197,0.005754,30,...,Pauschal - 3 Raten,Start,Spezialist (Kerngeschäft),Digital & Technology,3,8,2,2,5481,34
1,Debitor 10,1,2018-11-27,2018-11-27,1,0.042735,0.042735,0.000000,0.000000,1,...,Pauschalhonorar,Erfolgsauftrag,Spezialist (Kerngeschäft),Digital & Technology,1,1,1,1,0,2591
2,Debitor 100,36,2017-11-15,2025-09-26,7,2.910427,0.080845,0.338704,0.009408,9,...,Prozent - 3 Raten,Start,Spezialist (Kerngeschäft),Financial Services,2,4,4,2,2872,96
3,Debitor 1000,2,2012-10-31,2013-03-21,2,0.153846,0.076923,0.002564,0.001282,1,...,Prozent mit Anzahlung,Abschluss,Vertrieb/Presales/Marketing,Digital & Technology,1,2,1,1,141,4668
4,Debitor 1001,4,2011-07-28,2011-11-15,1,0.307692,0.076923,0.000000,0.000000,0,...,Prozent - 3 Raten,Präsentation,C-Level,Financial Services,1,2,2,1,110,5160


In [12]:
# Erste Plausibilitätsprüfung zentraler numerischer Merkmale

debitor_features[
    [
        "invoice_count",
        "active_years_count",
        "net_index_sum",
        "placement_count",
        "placement_rate",
        "leadership_rate",
        "active_days",
        "recency_days",
        "contract_type_nunique",
        "revenue_subtype_nunique",
        "job_category_nunique",
        "industry_nunique"
    ]
].describe().round(3)

,invoice_count,active_years_count,net_index_sum,placement_count,placement_rate,leadership_rate,active_days,recency_days,contract_type_nunique,revenue_subtype_nunique,job_category_nunique,industry_nunique
count,1364.000,1364.000,1364.000,1364.000,1364.000,1364.000,1364.000,1364.000,1364.000,1364.000,1364.000,1364.000
mean,11.143,2.390,0.919,2.851,0.231,0.259,727.979,2092.298,1.600,3.433,1.679,1.173
std,20.704,1.951,1.562,5.882,0.206,0.378,1086.895,1673.240,0.997,1.815,0.981,0.467
min,0.000,1.000,-0.071,0.000,0.000,0.000,0.000,0.000,1.000,1.000,1.000,1.000
25%,3.000,1.000,0.199,0.000,0.000,0.000,89.000,649.750,1.000,2.000,1.000,1.000
50%,5.000,2.000,0.413,1.000,0.231,0.000,260.000,1645.000,1.000,3.000,1.000,1.000
75%,11.000,3.000,0.971,3.000,0.333,0.500,854.000,3360.500,2.000,4.000,2.000,1.000
max,257.000,16.000,18.944,72.000,1.000,1.000,5712.000,5826.000,7.000,12.000,7.000,5.000


In [13]:
# Debitor-Matrix als Übergabestand für das nächste Notebook speichern

debitor_features.to_pickle(ARTIFACT_DIR / "01_debitor_features.pkl")

print("Gespeichert:", ARTIFACT_DIR / "01_debitor_features.pkl")

Gespeichert: c:\Users\jensm\Documents\data01-executive-search-analytics\notebooks\notebook_artifacts\01_debitor_features.pkl
